# 論文用図生成

`results/comparison/<dataset>/comparison_summary.csv` を読み込み，
3特徴量の中央値比較図を `results/paper_figures/<dataset>/` に保存する。

In [ ]:
from pathlib import Path

DATASET = "202604081400"

PROJECT_ROOT = None
FORMATS = ["pdf", "png"]
OVERWRITE = False

MIN_PREFIX_FLOW_COUNT = 1000
MAX_PREFIXES_TO_PLOT = 10
SORT_BY = "flow_count"
SAVE_LOG_SCALE_VERSION = True

: 

In [ ]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.labelsize": 13,
    "axes.titlesize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,
})


def find_project_root(dataset, configured_root=None):
    if configured_root is not None:
        root = Path(configured_root).expanduser().resolve()
        summary = root / "results" / "comparison" / dataset / "comparison_summary.csv"
        if not summary.exists():
            raise FileNotFoundError(f"comparison_summary.csv not found: {summary}")
        return root

    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        summary = candidate / "results" / "comparison" / dataset / "comparison_summary.csv"
        if summary.exists():
            return candidate

    raise FileNotFoundError(
        f"results/comparison/{dataset}/comparison_summary.csv was not found "
        f"under {start} or its parent directories."
    )


def relative_path(path, root):
    try:
        return str(Path(path).resolve().relative_to(root))
    except ValueError:
        return str(path)


def target_column(df):
    candidates = ["target", "normalized_dst_prefix", "dst_prefix", "prefix", "aggregate_id"]
    for column in candidates:
        if column in df.columns:
            return column
    lowered = {column.lower(): column for column in df.columns}
    for column in candidates:
        if column in lowered:
            return lowered[column]
    raise ValueError(f"target column not found. columns={list(df.columns)}")


def normalize_prefix_label(label):
    text = str(label).strip()
    if text.lower() == "overall":
        return "overall"

    text = re.sub(r"^(src|dst)_", "", text)
    if "/" in text:
        return text

    ipv4 = re.fullmatch(r"((?:\d{1,3}\.){3}\d{1,3})_(\d{1,2})", text)
    if ipv4:
        return f"{ipv4.group(1)}/{ipv4.group(2)}"

    ipv6 = re.fullmatch(r"([0-9A-Fa-f:]+)_(\d{1,3})", text)
    if ipv6 and ":" in ipv6.group(1):
        return f"{ipv6.group(1)}/{ipv6.group(2)}"

    sanitized_ipv6 = re.fullmatch(r"([0-9A-Fa-f_]+)___(\d{1,3})", text)
    if sanitized_ipv6:
        addr = sanitized_ipv6.group(1).replace("_", ":") + "::"
        return f"{addr}/{sanitized_ipv6.group(2)}"

    return text


def prefix_letter_label(index):
    alphabet = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    label = ""
    value = int(index)
    while True:
        value, remainder = divmod(value, len(alphabet))
        label = alphabet[remainder] + label
        if value == 0:
            return label
        value -= 1


def anonymize_display_labels(df):
    out = df.copy()
    labels = []
    prefix_index = 0
    for raw_label in out["prefix"].astype(str):
        if raw_label.lower() == "overall":
            labels.append("overall")
        else:
            labels.append(prefix_letter_label(prefix_index))
            prefix_index += 1
    out["display_label"] = labels
    return out


def load_summary(summary_csv, min_flow_count):
    df = pd.read_csv(summary_csv)

    rename_map = {
        "median_duration": "duration_median",
        "median_packet_count": "packet_count_median",
        "median_byte_count": "byte_count_median",
        "median_avg_packet_size": "avg_packet_size_median",
    }
    df = df.rename(columns={old: new for old, new in rename_map.items() if old in df.columns and new not in df.columns})

    numeric_columns = [
        "flow_count",
        "packet_count_median",
        "byte_count_median",
        "duration_median",
        "avg_packet_size_median",
        "tcp_ratio",
        "udp_ratio",
        "tcp_flow_ratio",
        "udp_flow_ratio",
    ]
    for column in numeric_columns:
        if column in df.columns:
            df[column] = pd.to_numeric(df[column], errors="coerce")

    tcol = target_column(df)

    overall_df = df[df[tcol].astype(str).str.lower() == "overall"].copy()
    if overall_df.empty:
        raise ValueError("overall row not found in comparison_summary.csv")
    overall_df["prefix"] = "overall"

    prefix_df = df[df[tcol].astype(str).str.lower() != "overall"].copy()
    prefix_df = prefix_df.rename(columns={tcol: "prefix"})
    if "flow_count" in prefix_df.columns:
        prefix_df = prefix_df[prefix_df["flow_count"].fillna(0) >= min_flow_count]
    prefix_df["display_label"] = prefix_df["prefix"].map(normalize_prefix_label)

    return overall_df.reset_index(drop=True), prefix_df.reset_index(drop=True)


def output_path(outdir, stem, ext, overwrite):
    path = outdir / f"{stem}.{ext.lstrip('.')}"
    if overwrite or not path.exists():
        return path
    for index in range(1, 1000):
        candidate = outdir / f"{stem}_{index:02d}.{ext.lstrip('.')}"
        if not candidate.exists():
            return candidate
    raise FileExistsError(f"too many existing output files: {path}")


def save_figure(fig, outdir, stem, formats, overwrite):
    outdir.mkdir(parents=True, exist_ok=True)
    saved = []
    for fmt in formats:
        path = output_path(outdir, stem, fmt, overwrite)
        fig.savefig(path, bbox_inches="tight", pad_inches=0.12)
        saved.append(path)
    return saved

In [ ]:
def plot_median_comparison(overall_df, prefix_df, output_dir, max_prefixes, sort_by, use_log_y):
    metrics = ["packet_count_median", "byte_count_median", "duration_median"]
    missing = [metric for metric in metrics if metric not in pd.concat([overall_df, prefix_df]).columns]
    if missing:
        raise ValueError(f"required columns are missing: {missing}")

    if sort_by not in prefix_df.columns:
        sort_by = "flow_count" if "flow_count" in prefix_df.columns else metrics[0]

    work_prefix = prefix_df.copy()
    work_prefix[sort_by] = pd.to_numeric(work_prefix[sort_by], errors="coerce")
    work_prefix = work_prefix.sort_values(sort_by, ascending=False).head(max_prefixes)

    plot_df = pd.concat([overall_df.iloc[[0]], work_prefix], ignore_index=True, sort=False)
    plot_df = anonymize_display_labels(plot_df)

    x = np.arange(len(plot_df))
    colors = ["#5B5B5B"] + ["#4C78A8"] * (len(plot_df) - 1)

    ylabels = {
        "packet_count_median": "Median packets per flow",
        "byte_count_median": "Median bytes per flow",
        "duration_median": "Median flow duration [s]",
    }
    titles = {
        "packet_count_median": "Packet count",
        "byte_count_median": "Byte count",
        "duration_median": "Duration",
    }

    fig, axes = plt.subplots(1, 3, figsize=(11.4, 3.9))
    for ax, metric in zip(axes, metrics):
        values = pd.to_numeric(plot_df[metric], errors="coerce")
        ax.bar(x, values, color=colors, edgecolor="white", linewidth=0.6)
        ax.set_title(titles[metric])
        ax.set_ylabel(ylabels[metric])
        ax.set_xticks(x)
        ax.set_xticklabels(plot_df["display_label"], rotation=35, ha="right")
        ax.set_xlabel("prefix", labelpad=-14)
        ax.margins(x=0.02)

        if use_log_y and (values > 0).any():
            ax.set_yscale("log")
            ax.set_ylabel(f"{ylabels[metric]} (log scale)")

    fig.align_ylabels(axes)
    fig.tight_layout(pad=1.1, w_pad=2.4)
    fig.subplots_adjust(left=0.09, bottom=0.34)

    suffix = "_logy" if use_log_y else ""
    saved = save_figure(
        fig,
        output_dir,
        f"prefix_median_packet_byte_duration_comparison{suffix}",
        FORMATS,
        OVERWRITE,
    )

    plt.show()

    return plot_df, saved

In [ ]:
project_root = find_project_root(DATASET, PROJECT_ROOT)
summary_csv = project_root / "results" / "comparison" / DATASET / "comparison_summary.csv"
output_dir = project_root / "results" / "paper_figures" / DATASET

overall_summary, prefix_summaries = load_summary(summary_csv, MIN_PREFIX_FLOW_COUNT)

print(f"dataset: {DATASET}")
print(f"input: {relative_path(summary_csv, project_root)}")
print(f"output: {relative_path(output_dir, project_root)}")
print(f"prefix rows: {len(prefix_summaries)}")

plot_df, saved_paths = plot_median_comparison(
    overall_summary,
    prefix_summaries,
    output_dir,
    MAX_PREFIXES_TO_PLOT,
    SORT_BY,
    use_log_y=False,
)

if SAVE_LOG_SCALE_VERSION:
    _, log_saved_paths = plot_median_comparison(
        overall_summary,
        prefix_summaries,
        output_dir,
        MAX_PREFIXES_TO_PLOT,
        SORT_BY,
        use_log_y=True,
    )
    saved_paths += log_saved_paths

for path in saved_paths:
    print("saved:", relative_path(path, project_root))